# ScenePatch — Gemma 4 E2B reproducibility notebook

This notebook is the primary executable Gemma path for ScenePatch: a controlled synthetic before/after fixture pack with generated spoken-intent audio, the official `google/gemma-4-E2B-it` checkpoint, three native function declarations, and a deterministic validator that never dynamically executes a model-selected function. Public-use approval for the exact five media hashes is recorded in `docs/FIXTURE_RIGHTS.md`.

**Why the notebook is primary:** on 31 July 2026, the pinned browser q4f16 runtime loaded and emitted native calls on the release M4 Pro, but it proposed a commit for the controlled missing-blue-marker fixture. That fails ScenePatch's release gate. The public Pages UI must therefore remain a clearly labeled scripted fixture replay unless a later tagged browser build passes the complete gate.

**No benchmark result is embedded here.** The notebook records only values produced by the current execution. The public-use approval in `docs/FIXTURE_RIGHTS.md` applies only while the five media hashes remain unchanged.

### Kaggle setup

1. Attach Kaggle's official Google Gemma 4 Transformers `gemma-4-e2b-it` V1 model input and enable a GPU accelerator. Internet is used only to install the Python dependencies, with Transformers pinned to 5.14.1; the attached model weights load from Kaggle input storage with `local_files_only=True`.
2. Attach the controlled-synthetic dataset named `scenepatch-controlled-fixture` containing `scene-before.png`, `scene-after-bad.png`, `scene-after-corrected.png`, `scene-after-occluded.png`, and `intent.wav`. The exact five media hashes are approved for the repository, Kaggle dataset/notebook, and demo video; regenerated or edited media require new approval.
3. Run all cells. The model loads once, then the notebook preprocesses and records the bad, corrected, and occluded cases separately. The full checkpoint is large, so this is a transparent reproducibility path rather than a browser-latency claim.

References: [Gemma 4 E2B IT model card](https://huggingface.co/google/gemma-4-E2B-it) and [Gemma 4 function calling](https://ai.google.dev/gemma/docs/capabilities/text/function-calling-gemma4).

In [ ]:
%pip install -q -U "transformers==5.14.1" accelerate librosa soundfile jsonschema

In [ ]:
from __future__ import annotations

from collections.abc import Mapping
from pathlib import Path
import hashlib
import json
import platform
import re
import time
import unicodedata

import kagglehub
import librosa
import soundfile as sf
import torch
import transformers
from PIL import Image, ImageChops, ImageDraw, ImageFilter, ImageFont, ImageOps
from IPython.display import Audio, display
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
MODEL_REVISION = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"
KAGGLE_MODEL_HANDLE = "google/gemma-4/transformers/gemma-4-e2b-it/1"
MODEL_IDENTIFIER = f"kaggle://{KAGGLE_MODEL_HANDLE}"
UPSTREAM_REFERENCE = f"{MODEL_ID}@{MODEL_REVISION} (documented reference; not byte-verified against Kaggle V1)"
MAX_AUDIO_SECONDS = 10.0
SAMPLE_RATE = 16_000

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before loading the full checkpoint.")
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model_id": MODEL_IDENTIFIER,
    "upstream_reference": UPSTREAM_REFERENCE,
})

## Native tool declarations and fail-closed validator

Gemma proposes calls; the code below only validates data. It does not use `eval`, `globals()`, imports, shell commands, network calls, or any function name supplied by the model. A parse or validator failure receives exactly one constrained correction attempt; a valid semantic decision is never retried, and a second malformed result remains blocked. A clean proposal remains pending until a human confirms it in the application.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "record_change",
            "description": "Record one visible scene change relative to the spoken intent.",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {
                        "type": "string",
                        "description": "Name one actual changed object or occlusion; never use only a region label or describe an unchanged object.",
                    },
                    "classification": {
                        "type": "string",
                        "enum": ["intended", "unexplained", "uncertain"],
                        "description": "How this change relates to the spoken intent.",
                    },
                },
                "required": ["description", "classification"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "commit_patch",
            "description": "Propose a patch only when every visible change is intended.",
            "parameters": {
                "type": "object",
                "properties": {"summary": {"type": "string"}},
                "required": ["summary"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "block_commit",
            "description": "Propose a block when any change is unexplained or uncertain.",
            "parameters": {
                "type": "object",
                "properties": {"reason": {"type": "string"}},
                "required": ["reason"],
                "additionalProperties": False,
            },
        },
    },
]

ALLOWED_NAMES = {"record_change", "commit_patch", "block_commit"}
CLASSIFICATIONS = {"intended", "unexplained", "uncertain"}


def _object_arguments(value):
    if isinstance(value, Mapping):
        return dict(value)
    if isinstance(value, str):
        decoded = json.loads(value)
        if isinstance(decoded, Mapping):
            return dict(decoded)
    raise ValueError("Tool arguments must be a JSON object.")


def normalize_tool_calls(parsed_response):
    if not isinstance(parsed_response, Mapping):
        raise ValueError("Parsed response must be an object.")
    raw_calls = parsed_response.get("tool_calls")
    if not isinstance(raw_calls, list):
        raise ValueError("Response must contain a tool_calls list.")

    normalized = []
    for item in raw_calls:
        if not isinstance(item, Mapping):
            raise ValueError("Each tool call must be an object.")
        payload = item.get("function", item)
        if not isinstance(payload, Mapping):
            raise ValueError("Each function payload must be an object.")
        name = payload.get("name")
        if name not in ALLOWED_NAMES:
            raise ValueError(f"Unknown tool: {name!r}")
        normalized.append({"name": name, "arguments": _object_arguments(payload.get("arguments"))})
    return normalized


def require_tool_only_response(parsed_response):
    for field in ("content", "reasoning", "reasoning_content"):
        value = parsed_response.get(field)
        if value not in (None, "", []):
            raise ValueError(f"Tool response must not contain non-tool {field}.")


def _bounded_text(value, field, maximum):
    if not isinstance(value, str):
        raise ValueError(f"{field} must be a string.")
    cleaned = " ".join(value.split())
    if not 1 <= len(cleaned) <= maximum:
        raise ValueError(f"{field} must contain 1 to {maximum} characters.")
    return cleaned


def _semantic_change_key(description):
    return re.sub(r"[.!?]+$", "", unicodedata.normalize("NFKC", description)).casefold()


SEMANTIC_SIGNAL_ALIASES = {
    "red_marker": (r"\b(?:red|crimson|scarlet) (?:marker|pen)\b", r"\b(?:marker|pen) (?:is )?(?:red|crimson|scarlet)\b"),
    "blue_marker": (r"\b(?:blue|navy|cobalt|azure) (?:marker|pen)\b", r"\b(?:marker|pen) (?:is )?(?:blue|navy|cobalt|azure)\b"),
    "occlusion": (r"\bocclu\w*\b", r"\bobscur\w*\b", r"\bcover(?:ed|ing)?\b", r"\bhidden\b"),
}


def semantic_signals_in_text(description):
    normalized = re.sub(r"_+", " ", unicodedata.normalize("NFKC", description).casefold())
    return {
        signal
        for signal, patterns in SEMANTIC_SIGNAL_ALIASES.items()
        if any(re.search(pattern, normalized) for pattern in patterns)
    }


def validate_record_change(call, existing_changes, required_signals=()):
    if call["name"] != "record_change":
        raise ValueError("Expected record_change.")
    if len(existing_changes) >= 6:
        raise ValueError("At most six change records are allowed.")
    arguments = call["arguments"]
    if set(arguments) != {"description", "classification"}:
        raise ValueError("record_change has missing or extra arguments.")
    description = re.sub(r"_+", " ", _bounded_text(arguments["description"], "description", 280))
    classification = arguments["classification"]
    if classification not in CLASSIFICATIONS:
        raise ValueError(f"Invalid classification: {classification!r}")
    duplicate_key = _semantic_change_key(description)
    existing_keys = {_semantic_change_key(item["description"]) for item in existing_changes}
    if duplicate_key in existing_keys:
        raise ValueError("Duplicate change description.")
    no_change_pattern = r"\b(?:unchanged|unmoved|still present)\b|\b(?:did|does|has) not move\b|\bnot moved\b|\b(?:kept|stayed|remained)\s+(?:put|present|unchanged|unmoved|in (?:the )?(?:same )?place)\b|\bno (?:visible )?change\b"
    if re.search(no_change_pattern, description, flags=re.IGNORECASE):
        raise ValueError("record_change must describe a change, not an unchanged object.")
    required = set(required_signals)
    matched = semantic_signals_in_text(description) & required
    if required and not matched:
        raise ValueError(f"description must name one required changed signal: {', '.join(sorted(required))}.")
    if len(matched) > 1:
        raise ValueError("record_change must describe exactly one required changed signal.")
    semantic_signal = next(iter(matched), None)
    if semantic_signal and semantic_signal in {item.get("semantic_signal") for item in existing_changes}:
        raise ValueError(f"Duplicate semantic signal: {semantic_signal}.")
    return {"description": description, "classification": classification, "semantic_signal": semantic_signal}


def require_semantic_coverage(changes, required_signals):
    required = set(required_signals)
    recorded = {change.get("semantic_signal") for change in changes}
    missing = sorted(required - recorded)
    if missing:
        raise ValueError(f"Terminal call rejected; uncovered semantic signal(s): {', '.join(missing)}.")
    return {"required": sorted(required), "recorded": sorted(recorded & required)}


def validate_terminal_call(call):
    name, arguments = call["name"], call["arguments"]
    if name not in {"commit_patch", "block_commit"}:
        raise ValueError("Expected commit_patch or block_commit.")
    expected = {"summary"} if name == "commit_patch" else {"reason"}
    if set(arguments) != expected:
        raise ValueError(f"{name} has missing or extra arguments.")
    field = next(iter(expected))
    return {"name": name, field: _bounded_text(arguments[field], field, 500)}


def decide_validated_sequence(changes, terminal, required_signals=(), intended_signals=()):
    if not 1 <= len(changes) <= 6:
        raise ValueError("One to six change records are required.")
    coverage = require_semantic_coverage(changes, required_signals)
    intended = set(intended_signals)
    unexpected = sorted(set(required_signals) - intended)
    unsafe = [c for c in changes if c["classification"] in {"unexplained", "uncertain"}]
    if unsafe or unexpected:
        decision = "blocked_by_deterministic_policy"
    elif terminal["name"] == "block_commit":
        decision = "blocked_as_proposed"
    else:
        decision = "pending_human_confirmation"
    coverage.update({"intended_by_fixture": sorted(intended), "unexpected": unexpected})
    return {"valid": True, "changes": changes, "terminal": terminal, "decision": decision, "semantic_coverage": coverage}


def validate_and_decide(parsed_response, required_signals=(), intended_signals=()):
    calls = normalize_tool_calls(parsed_response)
    if len(calls) > 7:
        raise ValueError("At most six changes and one terminal call are allowed.")
    changes, terminals = [], []
    for call in calls:
        if call["name"] == "record_change":
            changes.append(validate_record_change(call, changes, required_signals))
        else:
            terminals.append(validate_terminal_call(call))
    if len(terminals) != 1:
        raise ValueError("Exactly one terminal call is required.")
    return decide_validated_sequence(changes, terminals[0], required_signals, intended_signals)


In [ ]:
# Deterministic policy checks; these are code tests, not model-quality results.
clean_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Red marker moved above the sketchbook", "classification": "intended"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Requested marker move only"}}},
]}
unsafe_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Blue marker is missing", "classification": "intended"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Model proposed commit"}}},
]}
if validate_and_decide(clean_sample, ["red_marker"], ["red_marker"])["decision"] != "pending_human_confirmation":
    raise AssertionError("A clean covered proposal must remain pending human confirmation.")
if validate_and_decide(unsafe_sample, ["blue_marker"], ["red_marker"])["decision"] != "blocked_by_deterministic_policy":
    raise AssertionError("A visually required signal absent from the approved intent must override a commit.")
try:
    validate_and_decide({"tool_calls": [{"function": {"name": "delete_file", "arguments": {}}}]})
except ValueError as error:
    if "Unknown tool" not in str(error):
        raise
else:
    raise AssertionError("Unknown tools must fail closed.")
try:
    require_tool_only_response({"tool_calls": clean_sample["tool_calls"], "content": "extra prose"})
except ValueError as error:
    if "non-tool content" not in str(error):
        raise
else:
    raise AssertionError("Prose beside a tool call must fail closed.")
changed_record = validate_record_change(
    {"name": "record_change", "arguments": {"description": "Red marker moved", "classification": "intended"}},
    [],
    ["red_marker", "blue_marker"],
)
try:
    require_semantic_coverage([changed_record], ["red_marker", "blue_marker"])
except ValueError as error:
    if "blue_marker" not in str(error):
        raise
else:
    raise AssertionError("A red-only paraphrase must not cover the missing blue-marker signal.")
for unchanged_description in ("Sketchbook kept in place", "Blue marker still present", "Sketchbook did not move", "Yellow sticky note stayed put"):
    try:
        validate_record_change(
            {"name": "record_change", "arguments": {"description": unchanged_description, "classification": "intended"}},
            [],
            ["red_marker", "blue_marker"],
        )
    except ValueError as error:
        if "unchanged object" not in str(error):
            raise
    else:
        raise AssertionError("Unchanged-object language must not satisfy semantic coverage.")
try:
    validate_record_change(
        {"name": "record_change", "arguments": {"description": "R1 moved object", "classification": "intended"}},
        [],
        ["red_marker", "blue_marker"],
    )
except ValueError as error:
    if "required changed signal" not in str(error):
        raise
else:
    raise AssertionError("A region label without an object signal must fail closed.")
for underspecified_description, required_signal in (("Red thing moved", "red_marker"), ("Blue object missing", "blue_marker"), ("Card moved", "occlusion")):
    try:
        validate_record_change(
            {"name": "record_change", "arguments": {"description": underspecified_description, "classification": "intended"}},
            [],
            [required_signal],
        )
    except ValueError as error:
        if "required changed signal" not in str(error):
            raise
    else:
        raise AssertionError("Weak color or card aliases must not satisfy semantic coverage.")
underscore_record = validate_record_change(
    {"name": "record_change", "arguments": {"description": "Red_marker moved", "classification": "intended"}},
    [],
    ["red_marker"],
)
if underscore_record["description"] != "Red marker moved":
    raise AssertionError("Host-style underscored labels must normalize to readable text.")
try:
    validate_record_change(
        {"name": "record_change", "arguments": {"description": "Red marker absent from left side", "classification": "intended"}},
        [changed_record],
        ["red_marker", "blue_marker"],
    )
except ValueError as error:
    if "Duplicate semantic signal" not in str(error):
        raise
else:
    raise AssertionError("Two red-move descriptions must not satisfy red-plus-blue coverage.")
blue_record = validate_record_change(
    {"name": "record_change", "arguments": {"description": "Blue marker remained absent", "classification": "unexplained"}},
    [changed_record],
    ["red_marker", "blue_marker"],
)
require_semantic_coverage([changed_record, blue_record], ["red_marker", "blue_marker"])
occlusion_record = validate_record_change(
    {"name": "record_change", "arguments": {"description": "Occlusion", "classification": "uncertain"}},
    [],
    ["occlusion"],
)
require_semantic_coverage([occlusion_record], ["occlusion"])
print("Deterministic validator checks passed.")

## Load and normalize the controlled synthetic fixture

The next cell intentionally stops if the exact approved generated fixture pack is absent. It does not download or substitute other media, and it verifies the five media hashes before inference.

In [ ]:
fixture_names = {
    "scene-before.png",
    "scene-after-bad.png",
    "scene-after-corrected.png",
    "scene-after-occluded.png",
    "intent.wav",
}
fixture_roots = sorted({
    candidate.parent
    for candidate in Path("/kaggle/input").rglob("scene-before.png")
    if all((candidate.parent / name).is_file() for name in fixture_names)
})
if len(fixture_roots) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached ScenePatch fixture root under /kaggle/input; "
        f"found {len(fixture_roots)}: {[str(path) for path in fixture_roots]}"
    )
FIXTURE_ROOT = fixture_roots[0]
print({"fixture_root": str(FIXTURE_ROOT)})
BEFORE_PATH = FIXTURE_ROOT / "scene-before.png"
AFTER_CASES = {
    "bad": FIXTURE_ROOT / "scene-after-bad.png",
    "corrected": FIXTURE_ROOT / "scene-after-corrected.png",
    "occluded": FIXTURE_ROOT / "scene-after-occluded.png",
}
AUDIO_PATH = FIXTURE_ROOT / "intent.wav"

required_files = [BEFORE_PATH, *AFTER_CASES.values(), AUDIO_PATH]
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_hashes = {path.name: sha256_file(path) for path in required_files}
if len(input_hashes) != len(required_files):
    raise ValueError("Fixture filenames must be unique.")
print(json.dumps(input_hashes, indent=2))
APPROVED_HASHES = {
    "scene-before.png": "4822982cf8d254fa3b4579ab40c72131627c92a2780f77da7490ad865b7d83cf",
    "scene-after-bad.png": "71210ba4d515b1abe4a1c3e134e22fd9d9b2c7f1f1ea7b1396e96b155e2e2120",
    "scene-after-corrected.png": "9a22ca251072aee8d79d84ba4f4562f46400a5fec651bc3dae19ba4929e60d03",
    "scene-after-occluded.png": "b16ff8c9d41b65522fc0d72c8a9b108f72353c54315950fa4460c23e6370b5d3",
    "intent.wav": "d99fe8ce17149e9c8bc94e90467a95d3e1f5b98ba67e7fbc7f2c8fe9f201b020",
}
if input_hashes != APPROVED_HASHES:
    raise ValueError("Fixture hashes do not match the approved release record.")
print("All five media hashes match the approved public-use record.")

In [ ]:
WORK_ROOT = Path("/kaggle/working/scenepatch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
NORMALIZED_AUDIO = WORK_ROOT / "intent-16khz-mono.wav"

REGION_COLORS = ["#ff3b9d", "#ff9f1c", "#00c2ff", "#7cff6b", "#b692ff", "#ff5a5f"]
APPROVED_INTENT_TRANSCRIPT = "Move the red marker above the sketchbook. Keep everything else exactly where it is."
INTENDED_SEMANTIC_SIGNALS = sorted(semantic_signals_in_text(APPROVED_INTENT_TRANSCRIPT))


def _pixel_fraction(image, box, predicate):
    pixels = list(image.crop(box).getdata())
    return sum(1 for red, green, blue in pixels if predicate(red, green, blue)) / len(pixels)


def detect_region_semantic_signals(before, after, box):
    image_area = before.width * before.height
    box_area = (box[2] - box[0]) * (box[3] - box[1])
    light_neutral = lambda red, green, blue: min(red, green, blue) > 190 and max(red, green, blue) - min(red, green, blue) < 40
    neutral_before = _pixel_fraction(before, box, light_neutral)
    neutral_after = _pixel_fraction(after, box, light_neutral)
    if box_area / image_area >= 0.12 and neutral_after >= 0.75 and neutral_after - neutral_before >= 0.40:
        return ["occlusion"]

    color_tests = {
        "red_marker": lambda red, green, blue: red > 130 and red > green * 1.35 and red > blue * 1.20,
        "blue_marker": lambda red, green, blue: blue > 80 and blue > red * 1.25 and blue > green * 1.08,
    }
    signals = []
    for signal, predicate in color_tests.items():
        before_fraction = _pixel_fraction(before, box, predicate)
        after_fraction = _pixel_fraction(after, box, predicate)
        appeared_or_disappeared = min(before_fraction, after_fraction) <= 0.08 and max(before_fraction, after_fraction) >= 0.25
        if appeared_or_disappeared and abs(after_fraction - before_fraction) >= 0.20:
            signals.append(signal)
    return sorted(signals)


def detect_change_regions(before_path, after_path):
    with Image.open(before_path) as opened:
        before = ImageOps.exif_transpose(opened).convert("RGB")
    with Image.open(after_path) as opened:
        after = ImageOps.exif_transpose(opened).convert("RGB")
    if before.size != after.size:
        raise ValueError("Before and after images must be aligned to the same dimensions.")

    grid_width, grid_height = 160, 120
    difference = ImageChops.difference(before, after).convert("L")
    reduced = difference.resize((grid_width, grid_height), Image.Resampling.BOX)
    binary = reduced.point(lambda value: 255 if value > 12 else 0).filter(ImageFilter.MaxFilter(3))
    pixels = list(binary.getdata())
    seen = bytearray(grid_width * grid_height)
    components = []
    for start, value in enumerate(pixels):
        if not value or seen[start]:
            continue
        stack, seen[start], xs, ys = [start], 1, [], []
        while stack:
            index = stack.pop()
            y, x = divmod(index, grid_width)
            xs.append(x)
            ys.append(y)
            neighbors = (
                index - grid_width if y else -1,
                index + grid_width if y < grid_height - 1 else -1,
                index - 1 if x else -1,
                index + 1 if x < grid_width - 1 else -1,
            )
            for neighbor in neighbors:
                if neighbor >= 0 and pixels[neighbor] and not seen[neighbor]:
                    seen[neighbor] = 1
                    stack.append(neighbor)
        if len(xs) >= 8:
            components.append((min(xs), min(ys), max(xs) + 1, max(ys) + 1))

    components.sort(key=lambda box: (box[1], box[0]))
    if not 1 <= len(components) <= 6:
        raise ValueError(f"Expected one to six aligned change regions; found {len(components)}.")
    scale_x, scale_y = before.width / grid_width, before.height / grid_height
    regions = []
    for index, (left, top, right, bottom) in enumerate(components, start=1):
        box = [round(left * scale_x), round(top * scale_y), round(right * scale_x), round(bottom * scale_y)]
        semantic_signals = detect_region_semantic_signals(before, after, box)
        if not semantic_signals:
            raise ValueError(f"Candidate R{index} lacks a deterministic semantic signal; fail closed.")
        regions.append({
            "id": f"R{index}",
            "box": box,
            "semantic_signals": semantic_signals,
        })
    return regions


def render_panel(source_path, label, regions):
    with Image.open(source_path) as opened:
        source = ImageOps.exif_transpose(opened).convert("RGB")
    panel = Image.new("RGB", (512, 512), "#ded7c8")
    contained = ImageOps.contain(source, (512, 448), method=Image.Resampling.LANCZOS)
    offset_x = (512 - contained.width) // 2
    offset_y = 64 + (448 - contained.height) // 2
    panel.paste(contained, (offset_x, offset_y))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, 511, 63), fill="#101412")
    header_font = ImageFont.load_default(size=20)
    region_font = ImageFont.load_default(size=16)
    draw.text((18, 20), f"{label} · {len(regions)} REGIONS", fill="#d8ff63" if label.startswith("BEFORE") else "#8ad9ff", font=header_font)
    scale = contained.width / source.width
    for region_index, region in enumerate(regions):
        left, top, right, bottom = region["box"]
        mapped = (
            offset_x + round(left * scale),
            offset_y + round(top * scale),
            offset_x + round(right * scale),
            offset_y + round(bottom * scale),
        )
        color = REGION_COLORS[region_index]
        draw.rectangle(mapped, outline=color, width=5)
        badge = (mapped[0], max(64, mapped[1] - 24), mapped[0] + 30, max(88, mapped[1]))
        draw.rectangle(badge, fill=color)
        draw.text((badge[0] + 5, badge[1] + 3), region["id"], fill="#101412", font=region_font)
    return panel


def build_contact_sheet(before_path, after_path, output_path):
    regions = detect_change_regions(before_path, after_path)
    required_signals = sorted({signal for region in regions for signal in region["semantic_signals"]})
    sheet = Image.new("RGB", (1024, 512), "white")
    sheet.paste(render_panel(before_path, "BEFORE · BASE", regions), (0, 0))
    sheet.paste(render_panel(after_path, "AFTER · WORKTREE", regions), (512, 0))
    ImageDraw.Draw(sheet).line((512, 0, 512, 512), fill="#777777", width=2)
    if sheet.size != (1024, 512):
        raise ValueError("Contact sheet must be exactly 1024 by 512 pixels.")
    sheet.save(output_path, format="WEBP", quality=90, method=6)
    return {
        "path": str(output_path),
        "width": sheet.width,
        "height": sheet.height,
        "sha256": sha256_file(output_path),
        "candidate_regions": regions,
        "required_semantic_signals": required_signals,
    }


source_audio_info = sf.info(AUDIO_PATH)
if not 0 < source_audio_info.duration <= MAX_AUDIO_SECONDS:
    raise ValueError(f"The spoken intent must be non-empty and at most {MAX_AUDIO_SECONDS:.0f} seconds.")
waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
if waveform.size == 0:
    raise ValueError("Intent audio is empty.")
sf.write(NORMALIZED_AUDIO, waveform, SAMPLE_RATE, subtype="PCM_16")
audio_record = {
    "source_filename": AUDIO_PATH.name,
    "source_seconds": source_audio_info.duration,
    "processed_seconds": len(waveform) / SAMPLE_RATE,
    "sample_rate_hz": SAMPLE_RATE,
    "channels": 1,
    "source_sha256": input_hashes[AUDIO_PATH.name],
    "processed_sha256": sha256_file(NORMALIZED_AUDIO),
    "approved_fixture_transcript": APPROVED_INTENT_TRANSCRIPT,
    "intended_semantic_signals": INTENDED_SEMANTIC_SIGNALS,
}

case_preprocessing = {}
for case_name, after_path in AFTER_CASES.items():
    contact_sheet_path = WORK_ROOT / f"before-after-{case_name}.webp"
    case_preprocessing[case_name] = {
        "before": {"filename": BEFORE_PATH.name, "sha256": input_hashes[BEFORE_PATH.name]},
        "after": {"filename": after_path.name, "sha256": input_hashes[after_path.name]},
        "contact_sheet": build_contact_sheet(BEFORE_PATH, after_path, contact_sheet_path),
        "audio": audio_record,
    }
    print(f"Prepared case: {case_name}")
    print(json.dumps(case_preprocessing[case_name], indent=2))
    display(Image.open(contact_sheet_path))

expected_region_counts = {"bad": 3, "corrected": 2, "occluded": 3}
actual_region_counts = {name: len(item["contact_sheet"]["candidate_regions"]) for name, item in case_preprocessing.items()}
if actual_region_counts != expected_region_counts:
    raise AssertionError(f"Owned fixture region detector drifted: {actual_region_counts}")
expected_signal_sets = {"bad": ["blue_marker", "red_marker"], "corrected": ["red_marker"], "occluded": ["occlusion", "red_marker"]}
actual_signal_sets = {name: item["contact_sheet"]["required_semantic_signals"] for name, item in case_preprocessing.items()}
if actual_signal_sets != expected_signal_sets:
    raise AssertionError(f"Owned fixture semantic-signal detector drifted: {actual_signal_sets}")
print({"owned_fixture_candidate_regions": actual_region_counts})
print({"owned_fixture_semantic_signals": actual_signal_sets, "intended_semantic_signals": INTENDED_SEMANTIC_SIGNALS})

display(Audio(filename=str(NORMALIZED_AUDIO)))

## Load the official checkpoint

This is Kaggle's V1 copy of the unquantized official Google checkpoint and is separate from the browser's ONNX runtime. The attached model input resolves through Kaggle's local resource cache; no token is embedded and model loading is fail-closed with `local_files_only=True`.

In [ ]:
load_started = time.perf_counter()
MODEL_ROOT = Path(kagglehub.model_download(KAGGLE_MODEL_HANDLE))
for required_name in ("config.json", "processor_config.json", "tokenizer.json", "model.safetensors"):
    if not (MODEL_ROOT / required_name).is_file():
        raise FileNotFoundError(f"Attached Gemma model is missing {required_name}: {MODEL_ROOT}")
processor = AutoProcessor.from_pretrained(str(MODEL_ROOT), local_files_only=True)
model = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_ROOT),
    local_files_only=True,
    dtype="auto",
    device_map="auto",
)
model.eval()
model_load_seconds = time.perf_counter() - load_started
print({"model_id": MODEL_IDENTIFIER, "upstream_reference": UPSTREAM_REFERENCE, "model_root": str(MODEL_ROOT), "load_seconds_this_run": model_load_seconds, "device": str(model.device)})

In [ ]:
SYSTEM_PROMPT = """You are ScenePatch's visual change reviewer for a small creative desk.
Compare the labeled BEFORE and AFTER panels with the spoken intent. Colored R-number outlines and host-provided color/occlusion signals are deterministic change candidates, not intent classifications. Inspect every outlined region in both panels. A moved object can create two regions but is one semantic change; a missing object or an occluded area is a separate change. Record each required signal once, and never record unchanged objects.
Classify a requested change as intended, an unrequested change as unexplained, and ambiguous or occluded evidence as uncertain.
Call exactly one tool per turn. Obey the host's next_required_signal and terminal_allowed fields. First call record_change once for each material visible change, waiting for the deterministic tool acknowledgement after every call. Do not call a terminal tool until terminal_allowed is true. Then call exactly one commit_patch or block_commit. Commit only if every change is intended; otherwise block.
Do not make safety, identity, theft, inventory, or forensic claims."""

CORRECTION_INSTRUCTION = (
    "Correction: make exactly one valid native tool call now. Record one still-unrecorded material "
    "change using exactly description and classification. Name exactly one still-unrecorded host signal "
    "from the rejection reason; never use only an R label, repeat a recorded signal, or describe an unchanged object. "
    "Choose a terminal tool only after every required signal is recorded. "
    "Use no prose, undeclared arguments, or extra fields."
)


def initial_case_messages(preprocessing):
    contact_sheet = preprocessing["contact_sheet"]
    region_ids = [region["id"] for region in contact_sheet["candidate_regions"]]
    required_signals = contact_sheet["required_semantic_signals"]
    grounding_instruction = (
        f"Review all {len(region_ids)} outlined pixel-difference candidates ({', '.join(region_ids)}) "
        f"against the spoken intent. The host preflight requires coverage of: {', '.join(required_signals)}. "
        "Those labels identify visible color/occlusion evidence only; use the audio to determine whether each is "
        f"intended, unexplained, or uncertain. Call record_change exactly once now for the first signal, {required_signals[0]}; "
        "do not call a terminal yet. Record exactly one required signal per change. Use only the declared tools."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "path": str(contact_sheet["path"])},
                {"type": "text", "text": grounding_instruction},
                {"type": "audio", "audio": str(NORMALIZED_AUDIO)},
            ],
        },
    ]


def prepare_case_inputs(messages):
    return processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

In [ ]:
MAX_TOOL_GENERATIONS = 8  # six change records, one terminal call, and one global correction


def blocked_audit(stage, error, changes=None):
    return {
        "valid": False,
        "changes": list(changes or []),
        "terminal": None,
        "decision": "blocked_invalid_output",
        "failure": {"stage": stage, **error},
    }


def native_tool_exchange(call, response):
    # Gemma 4's pinned chat template expects the host result beside the assistant call.
    return {
        "role": "assistant",
        "tool_calls": [{
            "type": "function",
            "function": {"name": call["name"], "arguments": call["arguments"]},
        }],
        "tool_responses": [{"name": call["name"], "response": response}],
    }


def continuation_response(changes, required_signals):
    recorded_signals = {change.get("semantic_signal") for change in changes}
    missing_signals = sorted(set(required_signals) - recorded_signals)
    next_required_signal = missing_signals[0] if missing_signals else None
    if len(changes) >= 6:
        instruction = "Six changes are recorded. Call exactly one terminal tool now."
    elif missing_signals:
        instruction = (
            f"Still-unrecorded host semantic signal(s): {', '.join(missing_signals)}. The next_required_signal is "
            f"{next_required_signal}. Call record_change exactly once for that signal now. Use the audio to classify it; "
            "do not repeat a recorded signal, describe an unchanged object, or call a terminal."
        )
    else:
        instruction = (
            "Every host-required signal is recorded. Re-check the audio relationship, then call exactly one terminal tool."
        )
    return {
        "status": "recorded",
        "change_count": len(changes),
        "required_semantic_signals": sorted(required_signals),
        "recorded_semantic_signals": sorted(recorded_signals & set(required_signals)),
        "missing_semantic_signals": missing_signals,
        "next_required_signal": next_required_signal,
        "terminal_allowed": not missing_signals,
        "instruction": instruction,
    }


captured_intermediate = (
    '<|tool_call>call:record_change{classification:<|"|>intended<|"|>,'
    'description:<|"|>red marker moved above the sketchbook<|"|>}<tool_call|><|tool_response>'
)
captured_parsed = processor.parse_response(captured_intermediate)
captured_calls = normalize_tool_calls(captured_parsed)
if len(captured_calls) != 1 or captured_calls[0]["name"] != "record_change":
    raise AssertionError("The captured Kaggle handoff must parse as one intermediate record_change call.")
smoke_exchange = native_tool_exchange(captured_calls[0], {"status": "recorded"})
smoke_render = processor.apply_chat_template(
    [{"role": "user", "content": "Inspect."}, smoke_exchange],
    tools=TOOLS,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
if "<|tool_call>call:record_change" not in smoke_render or "<|tool_response>response:record_change" not in smoke_render:
    raise AssertionError("Gemma's native call/response exchange did not serialize with both handoff markers.")
print("Native Gemma tool-loop serialization checks passed.")


case_generations = {}
for case_name, preprocessing in case_preprocessing.items():
    messages = initial_case_messages(preprocessing)
    required_signals = preprocessing["contact_sheet"]["required_semantic_signals"]
    attempts, changes, accepted_calls = [], [], []
    correction_used = False
    next_kind = "initial"
    final_audit = None

    for attempt_number in range(1, MAX_TOOL_GENERATIONS + 1):
        attempt_kind = next_kind
        inputs = prepare_case_inputs(messages)
        input_length = inputs["input_ids"].shape[-1]

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_seconds = time.perf_counter() - inference_started

        generated_tokens = generated[0][input_length:]
        raw_output = processor.decode(generated_tokens, skip_special_tokens=False)
        parsed_output, call, failure, audit, tool_response = None, None, None, None, None

        try:
            parsed_output = processor.parse_response(raw_output)
        except Exception as error:
            failure = {"stage": "parse", "type": type(error).__name__, "message": str(error)}
        else:
            try:
                require_tool_only_response(parsed_output)
                calls = normalize_tool_calls(parsed_output)
                if len(calls) != 1:
                    raise ValueError("Each Gemma turn must contain exactly one tool call.")
                call = calls[0]
                if call["name"] == "record_change":
                    change = validate_record_change(call, changes, required_signals)
                    call = {"name": "record_change", "arguments": {
                        "description": change["description"],
                        "classification": change["classification"],
                    }}
                    changes.append(change)
                    accepted_calls.append(call)
                    tool_response = continuation_response(changes, required_signals)
                    messages.append(native_tool_exchange(call, tool_response))
                else:
                    terminal = validate_terminal_call(call)
                    require_semantic_coverage(changes, required_signals)
                    final_audit = decide_validated_sequence(list(changes), terminal, required_signals, INTENDED_SEMANTIC_SIGNALS)
                    terminal_field = "summary" if terminal["name"] == "commit_patch" else "reason"
                    call = {"name": terminal["name"], "arguments": {terminal_field: terminal[terminal_field]}}
                    accepted_calls.append(call)
                    audit = final_audit
                    tool_response = {"status": "accepted", "effective_decision": audit["decision"]}
                    messages.append(native_tool_exchange(call, tool_response))
            except Exception as error:
                failure = {"stage": "validate", "type": type(error).__name__, "message": str(error)}

        if failure is not None:
            audit = blocked_audit(failure["stage"], {"type": failure["type"], "message": failure["message"]}, changes)
            if correction_used or attempt_number == MAX_TOOL_GENERATIONS:
                final_audit = audit
            else:
                correction_used = True
                next_kind = "constrained_correction"
                if call is None:
                    messages.append({"role": "user", "content": CORRECTION_INSTRUCTION})
                else:
                    messages.append(native_tool_exchange(call, {
                        "status": "rejected",
                        "reason": failure["message"],
                        "instruction": CORRECTION_INSTRUCTION,
                    }))

        attempt_record = {
            "attempt": attempt_number,
            "kind": attempt_kind,
            "correction_instruction": CORRECTION_INSTRUCTION if attempt_kind == "constrained_correction" else None,
            "input_keys": sorted(inputs.keys()),
            "input_tokens": int(input_length),
            "generated_tokens": int(generated_tokens.shape[-1]),
            "inference_seconds_this_run": inference_seconds,
            "raw_output": raw_output,
            "parsed_output": parsed_output,
            "accepted_call": call if failure is None else None,
            "tool_response": tool_response,
            "changes_after_turn": list(changes),
            "failure": failure,
            "audit": audit,
        }
        attempts.append(attempt_record)

        print(f"\n=== CASE: {case_name} · TOOL TURN {attempt_number} ===")
        print("Raw Gemma generation:")
        print(raw_output)
        print("\nParsed turn state:")
        print(json.dumps({"parsed_output": parsed_output, "failure": failure, "changes": changes, "audit": audit}, indent=2, default=str))
        print({"inference_seconds_this_run": inference_seconds, "generated_tokens": int(generated_tokens.shape[-1])})
        del inputs, generated, generated_tokens

        if final_audit is not None:
            break
        if failure is None:
            next_kind = "native_continuation"

    if final_audit is None:
        limit_failure = {"type": "RuntimeError", "message": "No terminal tool call within the bounded generation loop."}
        final_audit = blocked_audit("limit", limit_failure, changes)
        attempts[-1]["failure"] = {"stage": "limit", **limit_failure}
        attempts[-1]["audit"] = final_audit

    selected_attempt = attempts[-1]
    case_generations[case_name] = {
        "attempts": attempts,
        "accepted_calls": accepted_calls,
        "changes": changes,
        "audit": final_audit,
        "correction_prompted": correction_used,
        "retry_used": any(attempt["kind"] == "constrained_correction" for attempt in attempts),
        "selected_attempt": selected_attempt["attempt"],
        "total_inference_seconds_this_run": sum(
            attempt["inference_seconds_this_run"] for attempt in attempts
        ),
        "total_generated_tokens": sum(attempt["generated_tokens"] for attempt in attempts),
    }

In [ ]:
# The final bounded tool turn is selected; malformed output receives at most one correction.
execution_records = {}
for case_name, preprocessing in case_preprocessing.items():
    generation = case_generations[case_name]
    attempts = generation["attempts"]
    selected = attempts[-1]
    execution_record = {
        "case": case_name,
        "model_id": MODEL_IDENTIFIER,
        "upstream_reference": UPSTREAM_REFERENCE,
        "model_load_seconds_this_run": model_load_seconds,
        "inference_seconds_this_run": selected["inference_seconds_this_run"],
        "total_inference_seconds_this_run": generation["total_inference_seconds_this_run"],
        "tool_turns": len(attempts),
        "total_generated_tokens": generation["total_generated_tokens"],
        "input_tokens": selected["input_tokens"],
        "generated_tokens": selected["generated_tokens"],
        "input_hashes": {
            preprocessing["before"]["filename"]: preprocessing["before"]["sha256"],
            preprocessing["after"]["filename"]: preprocessing["after"]["sha256"],
            AUDIO_PATH.name: input_hashes[AUDIO_PATH.name],
        },
        "preprocessing": preprocessing,
        "attempts": attempts,
        "accepted_calls": generation["accepted_calls"],
        "correction_prompted": generation["correction_prompted"],
        "retry_used": generation["retry_used"],
        "selected_attempt": generation["selected_attempt"],
        "raw_output": selected["raw_output"],
        "parsed_output": selected["parsed_output"],
        "audit": generation["audit"],
        "human_confirmation_performed": False,
    }
    execution_records[case_name] = execution_record
    record_path = WORK_ROOT / f"execution-{case_name}.json"
    record_path.write_text(json.dumps(execution_record, indent=2, default=str) + "\n", encoding="utf-8")
    print(f"\n=== AUDIT: {case_name} ===")
    print(json.dumps(execution_record, indent=2, default=str))
    print(f"Saved {record_path}")

print("Notebook complete. Each case has a separate execution record. A pending proposal is not a commit; confirmation remains a human application action.")

## Interpreting this run

One pass across the three cases demonstrates the mechanism, not model accuracy. Inspect `execution-bad.json`, `execution-corrected.json`, and `execution-occluded.json` separately. Before publishing a performance statement, run the release protocol on the same approved files and hardware: five consecutive clean-scene proposals and five consecutive bad-scene blocks, plus the occlusion case. Report every failure and the exact environment. Do not copy this notebook's full-checkpoint timing into the browser demo; they are different runtimes.